# 关于模型调用的invoke的使用

## 1. invoke的传参

### 1.1 文本输入

举例：

In [1]:
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv

# 加载配置文件
load_dotenv(override=True)

CLOSEAI_API_KEY = os.getenv("CLOSEAI_API_KEY")
CLOSEAI_BASE_URL = os.getenv("CLOSEAI_BASE_URL")

# 获取大模型
model = init_chat_model(
    model="openai:gpt-5.4-mini",
    api_key=CLOSEAI_API_KEY,
    base_url=CLOSEAI_BASE_URL,
)

In [2]:
response = model.invoke("翻译如下的汉字：你好世界")
print(response)

content='“你好世界”翻译成英文是：**Hello, world**。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 15, 'total_tokens': 35, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 5, 'engine_ttft_ms': 44, 'engine_ttlt_ms': 148, 'pre_inference_ms': 90, 'service_tbt_ms': 6, 'service_ttft_ms': 293, 'service_ttlt_ms': 397, 'total_duration_ms': 314, 'user_visible_ttft_ms': 204}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DhE2SG1ZSoMccTqR7yjtkwt1j7IYv', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019e4040-a02e-7221-becd-22078710d842-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 15, 'output_tokens': 20, 'total_tokens': 35,

### 1.2 字典列表

举例1：

In [3]:
messages = [
    {"role":"system","content":"你是一个专业的数学老师"},
    {"role":"user","content":"帮我解释一下什么是斐波那契数列"}
]

response = model.invoke(messages)
print(response)

content='当然可以。\n\n**斐波那契数列**是一列有规律的数字序列，它的特点是：\n\n- 第 1 项是 **1**\n- 第 2 项是 **1**\n- 从第 3 项开始，**每一项都等于前两项的和**\n\n也就是：\n\n\\[\n1,\\ 1,\\ 2,\\ 3,\\ 5,\\ 8,\\ 13,\\ 21,\\ 34,\\ \\dots\n\\]\n\n比如：\n- 第 3 项：\\(1+1=2\\)\n- 第 4 项：\\(1+2=3\\)\n- 第 5 项：\\(2+3=5\\)\n- 第 6 项：\\(3+5=8\\)\n\n---\n\n### 它的递推关系可以写成：\n\\[\nF_1=1,\\quad F_2=1,\\quad F_n=F_{n-1}+F_{n-2}\\ (n\\ge 3)\n\\]\n\n---\n\n### 为什么它有名？\n斐波那契数列在数学、自然界和计算机科学里都很常见，比如：\n- 植物的花瓣排列\n- 松果、向日葵中的螺旋结构\n- 算法设计和程序优化\n\n---\n\n### 一个小例子\n如果前两项是 1 和 1，那么后面几项就是：\n- 1 + 1 = 2\n- 1 + 2 = 3\n- 2 + 3 = 5\n- 3 + 5 = 8\n\n所以整列就变成：\n**1, 1, 2, 3, 5, 8, ...**\n\n如果你愿意，我还可以进一步给你讲：\n1. **斐波那契数列的来源**\n2. **它和黄金分割的关系**\n3. **如何用代码生成斐波那契数列**' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 420, 'prompt_tokens': 30, 'total_tokens': 450, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0,

举例2：涉及多轮对话

In [4]:
messages = [
    {"role":"system","content":"你是一个专业的数学老师"},
    {"role":"user","content":"1 + 2 = ？"},
    {"role":"assistant","content":"3"},
    {"role":"user","content":"我刚才问了什么问题？"},
]

response = model.invoke(messages)
print(response)

content='你刚才问的是：“1 + 2 = ？”' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 43, 'total_tokens': 60, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 6, 'engine_ttft_ms': 35, 'engine_ttlt_ms': 137, 'pre_inference_ms': 68, 'service_tbt_ms': 6, 'service_ttft_ms': 163, 'service_ttlt_ms': 260, 'total_duration_ms': 201, 'user_visible_ttft_ms': 95}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DhE83mpkrfnCWcQQgnhjTsbJltuOa', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019e4045-ef1e-7c41-a46f-38f55409439a-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 43, 'output_tokens': 17, 'total_tokens': 60, 'input_token

举例3：如果不传递历史，AI 会"失忆"

In [5]:
messages1 = [
    {"role": "system", "content": "你是一个非常友好的AI助手"},
    {"role": "user", "content": "你好，我叫小明"},
]
# 第一次对话
response1 = model.invoke(messages1)
# 打印响应
print(f"AI的回复1：{response1.content}")

messages2 = [
    {"role": "user", "content": "我叫什么名字？"}
]
# 第二次对话
response2 = model.invoke(messages2)
# 打印响应
print(f"AI的回复2：{response2.content}")


AI的回复1：你好，小明，很高兴认识你！我是一个很友好的 AI 助手 😊  
有什么我可以帮你的吗？
AI的回复2：我还不知道你的名字。  
如果你愿意告诉我，我可以记住并用它来称呼你。


作为对比，传递记忆

In [6]:
conversation = [
    {"role": "system", "content": "你是一个非常友好的AI助手"},
    {"role": "user", "content": "你好，我叫小明"},
]

# 第1次对话
response1 = model.invoke(conversation)
print(f"AI的回复1：{response1.content}")


#添加记忆
conversation.append({"role":"assistant","content":response1.content})
conversation.append({"role":"user","content":"我叫什么名字？"})

# 第2次对话
response2 = model.invoke(conversation)
print(f"AI的回复2：{response2.content}")

AI的回复1：你好，小明！很高兴认识你 😊  
我是一个很友好的 AI 助手，有什么我可以帮你的吗？
AI的回复2：你叫小明。


### 1.3 消息对象列表

举例1：

In [7]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage(content="你是一个专业的数学老师"),
    HumanMessage(content="帮我解释一下什么是斐波那契数列"),
]

response = model.invoke(messages)
print(response)

content='当然可以。\n\n**斐波那契数列**是一组很有名的数列，规律是：\n\n- 第 1 项是 **1**\n- 第 2 项也是 **1**\n- 从第 3 项开始，**每一项都等于前两项之和**\n\n也就是：\n\n\\[\n1,\\ 1,\\ 2,\\ 3,\\ 5,\\ 8,\\ 13,\\ 21,\\ 34,\\ \\dots\n\\]\n\n例如：\n- 第 3 项 = 1 + 1 = 2\n- 第 4 项 = 1 + 2 = 3\n- 第 5 项 = 2 + 3 = 5\n- 第 6 项 = 3 + 5 = 8\n\n---\n\n### 用数学式表示\n如果把第 \\(n\\) 项记作 \\(F_n\\)，那么：\n\n\\[\nF_1 = 1,\\quad F_2 = 1\n\\]\n\n\\[\nF_n = F_{n-1} + F_{n-2}\\quad (n \\ge 3)\n\\]\n\n---\n\n### 这个数列有什么特点？\n斐波那契数列的增长方式很特别，前面增加得不快，但后面会越来越快。它在自然界、数学、计算机科学中都很常见，比如：\n\n- 向日葵花盘的排列\n- 松果、菠萝的螺旋结构\n- 程序设计中的递归问题\n\n---\n\n### 一个简单记忆方法\n你可以把它理解成：\n\n> “后一个数 = 前两个数加起来”\n\n如果你愿意，我还可以继续给你讲：\n1. **斐波那契数列为什么叫这个名字**\n2. **它和递归有什么关系**\n3. **怎么用程序求斐波那契数列**' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 402, 'prompt_tokens': 30, 'total_tokens': 432, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0

举例2:传递记忆

In [8]:
from langchain_core.messages import SystemMessage, HumanMessage,AIMessage

messages = [
    SystemMessage(content="你是一个专业的数学老师"),
    HumanMessage(content="帮我解释一下什么是斐波那契数列"),
]
# 第1次对话
response1 = model.invoke(messages)
print(f"AI的回复1：{response1.content}")


#添加记忆
messages.append(AIMessage(content=response1.content))
messages.append(HumanMessage(content="我刚才问了什么问题？"))

# 第2次对话
response2 = model.invoke(messages)
print(f"AI的回复2：{response2.content}")

AI的回复1：当然可以。

**斐波那契数列**是一个非常著名的数列，它的特点是：

- 第 1 项和第 2 项通常都设为 **1**
- 从第 3 项开始，**每一项都等于前两项之和**

也就是：

\[
1,\ 1,\ 2,\ 3,\ 5,\ 8,\ 13,\ 21,\ \dots
\]

比如：
- 第 3 项 = 1 + 1 = 2
- 第 4 项 = 1 + 2 = 3
- 第 5 项 = 2 + 3 = 5
- 第 6 项 = 3 + 5 = 8

---

### 通项规律
如果用 \(F_n\) 表示第 \(n\) 项，那么可以写成：

\[
F_1 = 1,\quad F_2 = 1
\]

\[
F_n = F_{n-1} + F_{n-2}\quad (n \ge 3)
\]

---

### 它为什么有名？
斐波那契数列在数学、自然界和计算机科学里都很常见，比如：

- 花朵的花瓣排列
- 松果、向日葵中的螺旋结构
- 算法和程序设计中的递归问题

---

### 一个小例子
如果从 1 和 1 开始，后面几项是：

- 1
- 1
- 1+1 = 2
- 1+2 = 3
- 2+3 = 5
- 3+5 = 8

所以整个数列不断“往后累加”。

如果你愿意，我还可以继续给你讲：
1. **斐波那契数列的来源**
2. **它和兔子繁殖问题的关系**
3. **怎么用代码生成斐波那契数列**
AI的回复2：你刚才问的是：

**“帮我解释一下什么是斐波那契数列”**


## 2. invoke的返回值

举例1：

In [9]:
response = model.invoke([HumanMessage(content="2 + 3 * 2 = ?")])

In [10]:
print(type(response))

<class 'langchain_core.messages.ai.AIMessage'>


In [11]:
print(response)

content='2 + 3 * 2 = **8**' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 15, 'total_tokens': 30, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 4, 'engine_ttft_ms': 31, 'engine_ttlt_ms': 85, 'pre_inference_ms': 83, 'service_tbt_ms': 8, 'service_ttft_ms': 274, 'service_ttlt_ms': 401, 'total_duration_ms': 310, 'user_visible_ttft_ms': 191}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DhF25TmPWRGQ7HsOeULgTnDEpb349', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019e407a-ea7f-7f01-b0d3-d23fdba5ec3f-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 15, 'output_tokens': 15, 'total_tokens': 30, 'input_token_

美化输出：

In [12]:
from rich import print as rprint

rprint(response)

AIMessage(
    content='2 + 3 * 2 = **8**',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 15,
            'prompt_tokens': 15,
            'total_tokens': 30,
            'completion_tokens_details': {
                'accepted_prediction_tokens': 0,
                'audio_tokens': 0,
                'reasoning_tokens': 0,
                'rejected_prediction_tokens': 0
            },
            'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
            'latency_checkpoint': {
                'engine_tbt_ms': 4,
                'engine_ttft_ms': 31,
                'engine_ttlt_ms': 85,
                'pre_inference_ms': 83,
                'service_tbt_ms': 8,
                'service_ttft_ms': 274,
                'service_ttlt_ms': 401,
                'total_duration_ms': 310,
                'user_visible_ttft_ms': 191
            }
        },
        'model_provider': 'openai',
        'model_name': 'gpt-5.4-mini-2026-03-17',
        'system_fingerprint': None,
        'id': 'chatcmpl-DhF25TmPWRGQ7HsOeULgTnDEpb349',
        'service_tier': 'default',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019e407a-ea7f-7f01-b0d3-d23fdba5ec3f-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 15,
        'output_tokens': 15,
        'total_tokens': 30,
        'input_token_details': {'audio': 0, 'cache_read': 0},
        'output_token_details': {'audio': 0, 'reasoning': 0}
    }
)

举例2：

In [13]:
response = model.invoke("用一句话解释什么是 AI")

# 1. 获取回复内容
print("AI 回复:", response.content)

# 2. 获取响应元数据
metadata = response.response_metadata
print(f"使用的模型: {metadata['model_name']}")
print(f"结束原因: {metadata['finish_reason']}")
print(f"模型提供商：{metadata['model_provider']}\n")

# 3. 获取 Token 使用情况
usage = metadata.get('token_usage', {})
print(f"输入 tokens: {usage.get('prompt_tokens')}")
print(f"输出 tokens: {usage.get('completion_tokens')}")
print(f"总计 tokens: {usage.get('total_tokens')}")

# 4. 获取消息 ID
print(f"消息 ID: {response.id}")

AI 回复: AI（人工智能）是让计算机像人一样学习、思考和完成任务的技术。
使用的模型: gpt-5.4-mini-2026-03-17
结束原因: stop
模型提供商：openai

输入 tokens: 13
输出 tokens: 26
总计 tokens: 39
消息 ID: lc_run--019e4081-87e8-7d83-907c-86591fdf8de9-0
